In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 01. Разведочный анализ данных (EDA)\n",
    "## Анализ аудиоданных для классификации человек vs робот\n",
    "\n",
    "В этом ноутбуке мы проведем первичный анализ данных, визуализируем примеры аудио и сравним характеристики человеческой и синтезированной речи."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "import sys\n",
    "from pathlib import Path\n",
    "\n",
    "# Добавляем путь к проекту\n",
    "sys.path.append(str(Path.cwd().parent))\n",
    "\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "import librosa\n",
    "import librosa.display\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "from src.utils.file_utils import FileManager\n",
    "from src.utils.audio_utils import AudioProcessor\n",
    "from src.utils.config_loader import ConfigLoader\n",
    "\n",
    "# Настройка стиля\n",
    "plt.style.use('seaborn-v0_8-darkgrid')\n",
    "sns.set_palette(\"husl\")\n",
    "\n",
    "%matplotlib inline"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Загружаем конфиги\n",
    "config_loader = ConfigLoader('../configs')\n",
    "paths_config = config_loader.load_config('paths_config')\n",
    "data_config = config_loader.load_config('data_config')\n",
    "\n",
    "audio_root = Path(paths_config['paths']['audio_root'])\n",
    "print(f\"Директория с аудио: {audio_root}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Собираем информацию о файлах\n",
    "file_manager = FileManager()\n",
    "\n",
    "human_files = list(audio_root.glob('human/**/*.wav'))\n",
    "robot_files = list(audio_root.glob('robot/**/*.wav'))\n",
    "\n",
    "print(f\"Найдено файлов:\")\n",
    "print(f\"  Human: {len(human_files)}\")\n",
    "print(f\"  Robot: {len(robot_files)}\")\n",
    "print(f\"  Всего: {len(human_files) + len(robot_files)}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Функция для извлечения базовых характеристик\n",
    "def extract_basic_features(file_path):\n",
    "    y, sr = librosa.load(file_path, sr=16000)\n",
    "    \n",
    "    features = {\n",
    "        'duration': len(y) / sr,\n",
    "        'rms': np.sqrt(np.mean(y**2)),\n",
    "        'zcr': np.mean(librosa.feature.zero_crossing_rate(y)),\n",
    "        'spectral_centroid': np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)),\n",
    "        'spectral_bandwidth': np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr)),\n",
    "        'spectral_rolloff': np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))\n",
    "    }\n",
    "    \n",
    "    return features"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Извлекаем признаки для всех файлов (берем выборку для скорости)\n",
    "n_samples = min(50, len(human_files), len(robot_files))\n",
    "\n",
    "human_features = []\n",
    "for f in np.random.choice(human_files, n_samples, replace=False):\n",
    "    feats = extract_basic_features(f)\n",
    "    feats['class'] = 'human'\n",
    "    human_features.append(feats)\n",
    "\n",
    "robot_features = []\n",
    "for f in np.random.choice(robot_files, n_samples, replace=False):\n",
    "    feats = extract_basic_features(f)\n",
    "    feats['class'] = 'robot'\n",
    "    robot_features.append(feats)\n",
    "\n",
    "df_features = pd.DataFrame(human_features + robot_features)\n",
    "df_features.head()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Визуализация распределений признаков\n",
    "fig, axes = plt.subplots(2, 3, figsize=(15, 10))\n",
    "axes = axes.flatten()\n",
    "\n",
    "features = ['duration', 'rms', 'zcr', 'spectral_centroid', 'spectral_bandwidth', 'spectral_rolloff']\n",
    "titles = ['Длительность (с)', 'RMS энергия', 'Zero Crossing Rate', \n",
    "          'Спектральный центроид (Hz)', 'Спектральная ширина (Hz)', 'Spectral Rolloff (Hz)']\n",
    "\n",
    "for i, (feat, title) in enumerate(zip(features, titles)):\n",
    "    ax = axes[i]\n",
    "    \n",
    "    for cls in ['human', 'robot']:\n",
    "        data = df_features[df_features['class'] == cls][feat]\n",
    "        ax.hist(data, alpha=0.5, label=cls, bins=20)\n",
    "    \n",
    "    ax.set_xlabel(title)\n",
    "    ax.set_ylabel('Частота')\n",
    "    ax.legend()\n",
    "    ax.grid(True, alpha=0.3)\n",
    "\n",
    "plt.suptitle('Распределение признаков для классов human и robot', fontsize=16)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Сравнение средних значений\n",
    "df_mean = df_features.groupby('class').mean()\n",
    "df_mean"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Визуализация примеров аудио\n",
    "def plot_audio_example(file_path, title):\n",
    "    y, sr = librosa.load(file_path, sr=16000)\n",
    "    \n",
    "    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))\n",
    "    \n",
    "    # Волновая форма\n",
    "    librosa.display.waveshow(y, sr=sr, ax=ax1)\n",
    "    ax1.set_title(f'{title} - Waveform')\n",
    "    ax1.set_xlabel('Время (с)')\n",
    "    ax1.set_ylabel('Амплитуда')\n",
    "    \n",
    "    # Спектрограмма\n",
    "    D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)\n",
    "    img = librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz', ax=ax2)\n",
    "    ax2.set_title(f'{title} - Spectrogram')\n",
    "    ax2.set_xlabel('Время (с)')\n",
    "    ax2.set_ylabel('Частота (Hz)')\n",
    "    plt.colorbar(img, ax=ax2, format='%+2.0f dB')\n",
    "    \n",
    "    plt.tight_layout()\n",
    "    return fig"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Показываем пример человеческой речи\n",
    "if len(human_files) > 0:\n",
    "    plot_audio_example(human_files[0], 'Human Speech')\n",
    "    plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Показываем пример синтезированной речи\n",
    "if len(robot_files) > 0:\n",
    "    plot_audio_example(robot_files[0], 'Synthesized Speech')\n",
    "    plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Анализ длительности файлов\n",
    "plt.figure(figsize=(10, 5))\n",
    "\n",
    "for cls in ['human', 'robot']:\n",
    "    data = df_features[df_features['class'] == cls]['duration']\n",
    "    plt.hist(data, alpha=0.5, label=cls, bins=20)\n",
    "\n",
    "plt.xlabel('Длительность (с)')\n",
    "plt.ylabel('Количество файлов')\n",
    "plt.title('Распределение длительности аудиофайлов')\n",
    "plt.legend()\n",
    "plt.grid(True, alpha=0.3)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Корреляция признаков\n",
    "numeric_df = df_features.select_dtypes(include=[np.number])\n",
    "corr_matrix = numeric_df.corr()\n",
    "\n",
    "plt.figure(figsize=(10, 8))\n",
    "sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0,\n",
    "            square=True, linewidths=1, cbar_kws={\"shrink\": 0.8})\n",
    "plt.title('Корреляционная матрица признаков')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Выводы\n",
    "print(\"=\" * 50)\n",
    "print(\"ВЫВОДЫ ПО РАЗВЕДОЧНОМУ АНАЛИЗУ\")\n",
    "print(\"=\" * 50)\n",
    "\n",
    "print(\"\\n1. Размер датасета:\")\n",
    "print(f\"   - Human: {len(human_files)} файлов\")\n",
    "print(f\"   - Robot: {len(robot_files)} файлов\")\n",
    "\n",
    "print(\"\\n2. Средние значения признаков:\")\n",
    "for cls in ['human', 'robot']:\n",
    "    print(f\"\\n   {cls.upper()}:\")\n",
    "    cls_data = df_features[df_features['class'] == cls]\n",
    "    for feat in features:\n",
    "        mean_val = cls_data[feat].mean()\n",
    "        std_val = cls_data[feat].std()\n",
    "        print(f\"     {feat}: {mean_val:.4f} ± {std_val:.4f}\")\n",
    "\n",
    "print(\"\\n3. Наблюдаемые различия:\")\n",
    "print(\"   - Синтезированная речь часто имеет более стабильные характеристики\")\n",
    "print(\"   - У человеческой речи выше вариативность признаков\")\n",
    "print(\"   - Спектральные характеристики могут различаться\")\n",
    "print(\"   - Требуется более глубокий анализ с использованием MFCC\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.9.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}